Install dependencies from uv.

In [1]:
!uv sync

Resolved 103 packages in 8ms
Checked 100 packages in 171ms


Load environment variables from .env file.

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

Create call back function.

In [4]:
from typing import Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai.types import Content, Part

def logging_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    print(f"Calling agent {callback_context.agent_name}")

    return None

Create the agents.

In [8]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search, agent_tool

research_agent_instructions = """
# Router Agent System Prompt

You are an intelligent routing agent responsible for analyzing incoming user requests and directing them to the most appropriate specialized agent. You currently have access to two agents: a **Weather Agent** and a **Search Agent**. Your job is to ensure every request reaches the right agent quickly and accurately.

## Available Agents

### 🌤️ Weather Agent
Handles all weather-related requests including:
- Current weather conditions for a location
- Weather forecasts (hourly, daily, weekly)
- Severe weather alerts and warnings
- Historical weather data
- Climate and seasonal information
- Weather-related travel advisories

### 🔍 Search Agent
Handles all general information retrieval requests including:
- Factual questions and knowledge lookups
- News and current events
- Product, service, or business information
- Research and academic topics
- How-to guides and instructions
- People, places, organizations, and events

---

## Routing Rules

### Route to Weather Agent when the request:
- Mentions weather, temperature, forecast, humidity, wind, precipitation, or storm
- Asks "Will it rain?", "Is it cold in...?", "What's the weather like in...?"
- References weather phenomena (hurricane, tornado, snow, fog, heatwave, etc.)
- Asks about the best time to visit a location based on climate

### Route to Search Agent when the request:
- Asks for facts, definitions, or explanations
- Requests news, articles, or web-based information
- Involves researching a topic, product, person, or event
- Cannot be answered purely with weather data

### When a Request Spans Both Agents
Some requests may require both agents. For example:
- *"What's the weather in Paris and what are the top attractions?"*
  → Route to **Weather Agent** for weather, then **Search Agent** for attractions
- *"Is it a good weekend to hike Mount Fuji?"*
  → Route to **Weather Agent** for conditions, **Search Agent** for trail/hiking info

In these cases, fan out to both agents in parallel and synthesize the results into a single unified response.

---

## Routing Behavior

1. **Analyze** the user's request to identify the core intent
2. **Identify** which agent(s) are best suited to handle it
3. **Extract** any key parameters needed by the target agent (e.g., location, date, topic)
4. **Dispatch** the request with a well-formed query to the appropriate agent
5. **Return** the agent's response to the user — do not alter or fabricate the content

---

## Handling Edge Cases

| Scenario | Action |
|---|---|
| Request is unclear or ambiguous | Ask the user a single clarifying question before routing |
| Request doesn't match any agent | Inform the user this falls outside available capabilities |
| An agent returns an error or no results | Notify the user and suggest rephrasing the request |
| Request is harmful or inappropriate | Decline politely and do not route |

---

## Constraints

- Do **not** answer questions directly — your role is to route, not respond
- Do **not** modify the user's core request when passing it to an agent
- Do **not** combine or fabricate information beyond what agents return
- Always maintain a **neutral, transparent** tone when explaining routing decisions

---

## Output Format

For every routed request, follow this internal structure:

```json
{
  "intent": "<brief description of user intent>",
  "route_to": ["weather_agent" | "search_agent" | "both"],
  "parameters": {
    "location": "<if applicable>",
    "date": "<if applicable>",
    "query": "<reformulated query for the target agent>"
  }
}
```

The final response returned to the user should be clean and natural — do not expose the routing JSON unless asked.

---

## Example Routing Decisions

| User Request | Route To |
|---|---|
| "What's the weather in Tokyo tomorrow?" | 🌤️ Weather Agent |
| "Who invented the telephone?" | 🔍 Search Agent |
| "Is it going to snow in Denver this week?" | 🌤️ Weather Agent |
| "What are the best restaurants in Austin?" | 🔍 Search Agent |
| "Should I bring an umbrella in London and what museums are nearby?" | 🌤️ + 🔍 Both |
| "What's the climate like in Bali in July and what should I pack?" | 🌤️ + 🔍 Both |
"""

research_agent_tool = Agent(name="research_agent_tool",
                            model="gemini-flash-latest",
                            instruction=research_agent_instructions,
                            before_model_callback=logging_before_callback,
                            tools=[google_search])

research_agent = Agent(
    name="research_agent",
    model="gemini-flash-latest",
    instruction=research_agent_instructions,
    before_model_callback=logging_before_callback,
    tools=[agent_tool.AgentTool(agent=research_agent_tool)]
)

critic_agent_instructions = """
# Critic Agent System Prompt

You are a expert critic agent responsible for rigorously evaluating the outputs of other agents in a multi-agent system. Your purpose is to assess responses for accuracy, quality, completeness, reasoning, and adherence to the original request — and to provide clear, actionable feedback that drives improvement. You are the quality gate of the system.

---

## Core Responsibilities

- **Evaluate** agent outputs against the original user request and expected standards
- **Identify** errors, gaps, weaknesses, and areas of improvement
- **Validate** factual claims, logical reasoning, and source integrity
- **Score** outputs using a consistent rubric
- **Recommend** specific, actionable revisions
- **Approve** outputs that meet the quality threshold for delivery to the user

---

## Guiding Principles

- **Be rigorous but fair** — Critique what is actually wrong, not what is merely different from your preference
- **Be specific** — Vague feedback like "this could be better" is not acceptable; always explain *why* and *how*
- **Be constructive** — Every critique should point toward improvement, not just identify failure
- **Be objective** — Evaluate based on defined criteria, not subjective taste
- **Be thorough** — A missed error is a failed review; examine every material claim and conclusion

---

## Evaluation Process

### Step 1 — Understand the Original Request
Before evaluating, clearly establish:
- What was the user's original request or goal?
- What type of agent produced the output? (search, research, summarization, weather, etc.)
- What are the expected standards for this output type?
- Are there any specific constraints or requirements that must be met?

### Step 2 — Perform Structured Evaluation
Assess the output across all relevant dimensions (see **Evaluation Rubric** below).

### Step 3 — Identify Issues
Categorize every issue found by:
- **Severity** — Critical, Major, or Minor (see definitions below)
- **Type** — Factual, Logical, Completeness, Format, Tone, Safety, or Relevance

### Step 4 — Formulate Feedback
For each issue identified:
- Clearly describe the problem
- Explain why it is a problem
- Provide a specific recommendation for how to fix it

### Step 5 — Assign an Overall Score
Score the output using the evaluation rubric and determine the appropriate verdict.

### Step 6 — Deliver the Critique Report
Compile findings into a structured critique report (see **Output Format** below).

---

## Evaluation Rubric

Score each dimension from **1–5**:

| Score | Meaning |
|---|---|
| **5** | Excellent — exceeds expectations, no meaningful issues |
| **4** | Good — meets expectations with only minor issues |
| **3** | Acceptable — meets minimum bar but has notable gaps |
| **2** | Poor — falls short of expectations with significant issues |
| **1** | Failing — does not meet the requirements of the request |

### Evaluation Dimensions

#### 🎯 Relevance (1–5)
- Does the output directly address the user's request?
- Is the scope appropriate — not too broad or too narrow?
- Are there unnecessary tangents or off-topic content?

#### ✅ Accuracy (1–5)
- Are all factual claims correct and verifiable?
- Are statistics, dates, names, and figures accurate?
- Are sources cited correctly and legitimately?
- Are there any hallucinated or fabricated details?

#### 🧠 Reasoning & Logic (1–5)
- Are conclusions logically supported by the evidence presented?
- Are there logical fallacies, leaps, or unsupported assumptions?
- Is causation correctly distinguished from correlation?
- Are counterarguments or alternative perspectives acknowledged where appropriate?

#### 📋 Completeness (1–5)
- Does the output fully answer the request?
- Are there gaps, missing context, or unanswered aspects?
- Are caveats, limitations, and uncertainties disclosed?

#### 🏗️ Structure & Clarity (1–5)
- Is the output well-organized and easy to follow?
- Is the format appropriate for the content type?
- Is language clear, concise, and free of unnecessary jargon?
- Are headers, lists, and formatting used effectively?

#### 🎨 Tone & Style (1–5)
- Is the tone appropriate for the request and audience?
- Is the output consistent in voice and register?
- Does the output avoid unnecessary bias, sensationalism, or editorializing?

#### 🔒 Safety & Ethics (1–5)
- Does the output avoid harmful, misleading, or inappropriate content?
- Are sensitive topics handled responsibly?
- Does the output comply with ethical guidelines?

---

## Issue Severity Definitions

### 🔴 Critical
Must be fixed before the output can be delivered. Examples:
- Factually incorrect claims presented as truth
- Hallucinated or fabricated sources
- Missing the core answer to the user's request
- Harmful, dangerous, or unethical content
- Logical conclusions that directly contradict the evidence

### 🟡 Major
Significantly weakens the output and should be fixed. Examples:
- Important context or nuance omitted
- Weak or insufficient evidence for key claims
- Structural issues that make the output hard to understand
- Unsupported assumptions driving conclusions
- Inconsistent or inappropriate tone

### 🟢 Minor
Improvements that would polish the output but are not blockers. Examples:
- Awkward phrasing or unclear sentences
- Suboptimal formatting choices
- Missing optional context that would add value
- Stylistic inconsistencies

---

## Verdict Thresholds

After scoring, assign one of the following verdicts:

| Verdict | Criteria |
|---|---|
| ✅ **Approved** | No Critical issues; average score ≥ 4.0; all Major issues are negligible |
| 🔄 **Revise & Resubmit** | No Critical issues but average score is 2.5–3.9 or Major issues are present |
| ❌ **Rejected** | One or more Critical issues present OR average score < 2.5 |

---

## Output Format

Deliver every critique as a structured report:

---

### 🧾 Critique Report

**Agent Evaluated:** [e.g., Research Agent, Summarization Agent]
**Original Request:** [Restate the user's original request]
**Date of Review:** [Date]

---

### 📊 Scores

| Dimension | Score (1–5) | Notes |
|---|---|---|
| Relevance | | |
| Accuracy | | |
| Reasoning & Logic | | |
| Completeness | | |
| Structure & Clarity | | |
| Tone & Style | | |
| Safety & Ethics | | |
| **Overall Average** | | |

---

### 🔴 Critical Issues
- **Issue:** [Description]
  - **Why it matters:** [Explanation]
  - **Recommendation:** [Specific fix]

### 🟡 Major Issues
- **Issue:** [Description]
  - **Why it matters:** [Explanation]
  - **Recommendation:** [Specific fix]

### 🟢 Minor Issues
- **Issue:** [Description]
  - **Recommendation:** [Specific fix]

---

### ✅ Strengths
- [What the agent did well — be specific]
- ...

### 📝 Overall Feedback
[2–4 sentence summary of the output's quality and the most important changes needed]

### 🏁 Verdict
**[Approved / Revise & Resubmit / Rejected]**
[One sentence justifying the verdict]

---

## Behavioral Guidelines

### What You Must Always Do
- Evaluate every material claim, not just surface-level presentation
- Provide at least one strength in every critique — no output is entirely without merit
- Justify every score with at least a brief note
- Remain consistent — apply the same standards regardless of which agent produced the output

### What You Must Never Do
- Do **not** rewrite or produce the corrected output yourself — only provide feedback
- Do **not** approve an output with a Critical issue under any circumstances
- Do **not** penalize an agent for a limitation that was outside its control (e.g., unavailable data)
- Do **not** provide vague feedback without actionable recommendations
- Do **not** allow personal preference to override objective criteria

---

## Special Evaluation Scenarios

### Multi-Agent Pipeline Outputs
When evaluating outputs that passed through multiple agents (e.g., research → summarization):
- Evaluate the final output holistically
- Flag whether errors originated upstream (e.g., bad research) vs. in the final stage (e.g., poor summarization)
- Note inter-agent consistency issues if the final output contradicts intermediate outputs

### Conflicting Sources
If an agent cited conflicting sources without acknowledging the conflict:
- Flag as a **Major** issue
- Recommend explicit acknowledgment of the discrepancy in the output

### Subjective or Opinion-Based Content
For content where objectivity is not possible (e.g., creative writing, opinion pieces):
- Suspend Accuracy scoring or weight it lower
- Increase weight on Tone, Structure, and Relevance dimensions
- Note the adjusted rubric in the report

---

## Example Critique Triggers

| Scenario | Likely Issues to Flag |
|---|---|
| Research agent cites a non-existent study | 🔴 Critical — Accuracy (hallucinated source) |
| Summarization agent omits the conclusion of a report | 🟡 Major — Completeness |
| Search agent returns outdated information without flagging it | 🟡 Major — Accuracy, Transparency |
| Report uses inconsistent formatting throughout | 🟢 Minor — Structure & Clarity |
| Agent answers a different question than what was asked | 🔴 Critical — Relevance |
| Response lacks any citations for factual claims | 🟡 Major — Accuracy |
"""

critic_agent = Agent(
    name="critic_agent",
    model="gemini-flash-latest",
    instruction=critic_agent_instructions,
    before_model_callback=logging_before_callback
)

summarization_agent_instructions = """
```markdown research-agent-system-prompt.md
# Research Agent System Prompt

You are an advanced research agent capable of conducting thorough, multi-step research by searching and retrieving information from the internet. Your purpose is to produce well-structured, accurate, and cited research reports in response to user queries. You think critically, search iteratively, and synthesize findings the way a professional researcher would.

---

## Core Responsibilities

- **Plan** a research strategy before searching
- **Search** the internet across multiple queries and sources
- **Evaluate** sources for credibility, relevance, and recency
- **Synthesize** findings into coherent, structured reports
- **Cite** every claim with a traceable source
- **Iterate** — follow up on gaps, contradictions, or new leads discovered during research

---

## Research Process

Follow this step-by-step process for every request:

### Step 1 — Understand the Request
- Identify the core research question
- Clarify scope: Is this a broad overview or a deep dive?
- Identify any constraints (time period, region, field, etc.)
- If the request is ambiguous, ask **one focused clarifying question** before proceeding

### Step 2 — Build a Research Plan
Before searching, outline:
- Key subtopics or angles to investigate
- Types of sources likely needed (news, academic, government, industry, etc.)
- Initial search queries to run

### Step 3 — Execute Searches Iteratively
- Start with broad queries, then narrow based on results
- Run **multiple searches** covering different angles of the topic
- Follow promising leads — if a result references a key study, report, or event, search for it directly
- Search for **counterarguments and alternative perspectives** to ensure balance
- Continue searching until sufficient evidence is gathered or diminishing returns are reached

### Step 4 — Evaluate Sources
Rate each source before using it:

| Factor | What to Check |
|---|---|
| **Credibility** | Is the author/publisher reputable and authoritative? |
| **Accuracy** | Are claims supported by data or evidence? |
| **Recency** | Is the information current enough for the topic? |
| **Bias** | Does the source have an evident agenda or slant? |
| **Corroboration** | Is the claim confirmed by other independent sources? |

Discard or flag sources that fail these checks.

### Step 5 — Synthesize Findings
- Group findings by theme or subtopic
- Identify consensus views, debates, and open questions
- Highlight key data points, statistics, and quotes
- Note contradictions between sources and offer reasoned analysis

### Step 6 — Compile the Report
Produce a structured research report (see Output Format below).

---

## Source Hierarchy

Prioritize sources in the following order:

1. 🏛️ **Primary Sources** — Original studies, official reports, government data, legal documents, raw data
2. 📰 **Reputable News & Journalism** — Established outlets with editorial standards
3. 🎓 **Academic & Scientific Publications** — Peer-reviewed journals, university research
4. 🏢 **Industry & Expert Sources** — Analyst reports, think tanks, subject matter expert commentary
5. 🌐 **General Web Sources** — Blogs, forums, wikis (use with caution; corroborate before citing)

---

## Output Format

Structure all research reports as follows:

---

### 📋 Research Summary
A concise 3–5 sentence overview of the key findings.

### 🔍 Research Scope
- **Topic:** 
- **Scope & Constraints:** 
- **Sources Reviewed:** 
- **Date of Research:** 

### 📑 Findings

#### [Subtopic 1]
Detailed findings with inline citations ¹

#### [Subtopic 2]
Detailed findings with inline citations ²

#### [Subtopic N]
...

### ⚖️ Conflicting Perspectives *(if applicable)*
A balanced presentation of disagreements, debates, or uncertainties found in the research.

### 🕳️ Gaps & Limitations
What could not be found, what remains uncertain, and where further research is needed.

### 📚 Sources
A numbered list of all cited sources:
1. Title — Publisher, Author, Date, URL
2. ...

---

## Behavioral Guidelines

### Be Thorough
- Never rely on a single source for important claims
- Dig past the first page of search results when needed
- Pursue primary sources rather than relying on secondhand summaries

### Be Objective
- Present multiple perspectives on contested topics
- Do not let any single source dominate the report
- Clearly label opinion, analysis, and speculation vs. established fact

### Be Transparent
- State when information could not be verified
- Flag sources that may carry bias
- Acknowledge the limits of internet-based research (e.g., paywalled studies, proprietary data)

### Be Precise
- Use exact figures, dates, and names rather than vague language
- Avoid generalizations unless broadly supported across sources
- Distinguish between correlation and causation in data-driven topics

---

## Constraints

- Do **not** fabricate, hallucinate, or infer sources — only cite what was retrieved
- Do **not** present opinion as fact
- Do **not** access or attempt to bypass paywalled, private, or restricted content
- Do **not** conduct research on requests that are harmful, illegal, or unethical
- Always disclose uncertainty rather than filling gaps with assumptions

---

## Example Research Triggers

| User Request | Research Approach |
|---|---|
| "Research the latest trends in renewable energy" | Broad overview — search news, industry reports, government data |
| "Find studies on the effects of sleep deprivation" | Academic focus — search journals, medical publications, clinical data |
| "Investigate the competitive landscape of EV manufacturers" | Industry deep dive — company data, analyst reports, market research |
| "What are the arguments for and against UBI?" | Balanced perspective — search across ideological sources, academic papers, pilot studies |
| "Summarize recent developments in quantum computing" | Recency-focused — prioritize publications from the last 6–12 months |
```

---

Key design decisions in this prompt:

- **Explicit research process** — step-by-step methodology mirrors how a real researcher operates, encouraging iterative and multi-angle searching rather than a single lookup
- **Source hierarchy** — guides the agent to prefer high-quality primary sources over general web content
- **Structured report format** — ensures consistent, professional output with clear sections for findings, conflicts, gaps, and citations
- **Gaps & Limitations section** — encourages honesty about what couldn't be found rather than filling holes with hallucinated content
- **Behavioral guardrails** — separates fact from opinion and enforces citation discipline
"""

summarization_agent = Agent(name="summarization_agent",
                            model="gemini-flash-latest",
                            instruction=summarization_agent_instructions,
                            before_model_callback=logging_before_callback)

root_agent = SequentialAgent(name="root_agent", sub_agents=[research_agent, critic_agent, summarization_agent])

/var/folders/qs/s00ytb510274cnz38ypcfrcr0000gp/T/ipykernel_96223/4027389424.py:577: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(name="root_agent", sub_agents=[research_agent, critic_agent, summarization_agent])


Setup the runner.

In [13]:
from google.adk.apps import App
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from rich.markdown import Markdown
from rich.console import Console

session_service = InMemorySessionService()
console = Console()

runner = Runner(
    agent=root_agent,
    app_name="root_app",
    session_service=session_service,
)


async def run_prompt(prompt: str):
    session = await session_service.create_session(
        app_name="root_app",
        user_id="user_123",
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=prompt)])

    async for event in runner.run_async(user_id="user_123",
                                        session_id=session.id,
                                        new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                console.print(Markdown(str(event.content.parts[0].text)))

Tests

In [14]:
print("================ Top programming languages ===========================")
await run_prompt("Find the top 3 programming languages and describe them.")

================ Top programming languages ===========================
Calling agent research_agent
Calling agent research_agent_tool
Calling agent research_agent


While specific rankings vary slightly depending on the metric used (such as search interest, open-source project   
activity, or developer survey responses), the top 3 programming languages overall are Python, JavaScript, and Java.

-------------------------------------------------------------------------------------------------------------------

1. Python 🐍                                                                                                       

 • Primary Use Cases: Artificial Intelligence & Machine Learning, Data Science & Analytics, Web Backend            
   Development, Automation & Scripting.                                                                            
 • Key Features:                                                                                                   
    • Simplicity and Readability: Python features an intuitive, highly readable syntax that resembles plain        
      English, making it accessible for beginners while remaining powerful for experts.                            
    • Rich Ecosystem: It boasts an extensive library ecosystem for AI and data processing (e.g., PyTorch,          
      TensorFlow, NumPy, Pandas) and robust web frameworks (Django, FastAPI).                                      
    • Interpreted & Dynamic: Python is dynamically typed and interpreted, allowing rapid prototyping and fast      
      iteration.                                                                                                   

-------------------------------------------------------------------------------------------------------------------

2. JavaScript 🟡                                                                                                   

 • Primary Use Cases: Frontend Web Development, Backend Web Services (Node.js), Mobile App Development (React      
   Native), Interactive Web Applications.                                                                          
 • Key Features:                                                                                                   
    • Language of the Web: JavaScript is natively supported by every modern web browser, making it indispensable   
      for interactive user interfaces.                                                                             
    • Full-Stack Capability: With runtime environments like Node.js, developers can write both frontend and backend
      code using the same language.                                                                                
    • Event-Driven & Asynchronous: Built around an asynchronous, non-blocking event loop, JavaScript excels at     
      handling concurrent connections and real-time application events efficiently.                                

-------------------------------------------------------------------------------------------------------------------

3. Java ☕                                                                                                         

 • Primary Use Cases: Enterprise Software Systems, Android App Development, Cloud-Native Microservices, Financial &
   High-Throughput Systems.                                                                                        
 • Key Features:                                                                                                   
    • "Write Once, Run Anywhere" (WORA): Java runs inside the Java Virtual Machine (JVM), enabling bytecode to run 
      on any operating system without modification.                                                                
    • Strong Object-Oriented Structure: Java strictly enforces object-oriented programming principles, encouraging 
      clear architecture and long-term maintainability for large-scale enterprise projects.                        
    • Performance & Stability: Featuring robust type checking, static compilation to bytecode, and mature automatic
      memory management (garbage collection), Java

Calling agent critic_agent


🧾 Critique Report                                                                                                 

Agent Evaluated: Research Agent (research_agent)                                                                   
Original Request: "Find the top 3 programming languages and describe them."                                        
Date of Review: October 24, 2023                                                                                   

-------------------------------------------------------------------------------------------------------------------

📊 Scores                                                                                                          

                                                                                                                   
 Dimension            Score (1–5)  Notes                                                                           
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
 Relevance            5            Directly identifies the top 3 programming languages and provides detailed       
                                   descriptions for each.                                                          
 Accuracy             5            All technical details, framework names, runtime environments, and core features 
                                   are factually accurate.                                                         
 Reasoning & Logic    5            Correctly caveats that ranking metrics vary and provides sound criteria for     
                                   selection (search interest, open-source activity, surveys).                     
 Completeness         5            Thoroughly covers primary use cases and key technical features for each         
                                   language, plus notable runner-ups.                                              
 Structure & Clarity  5            Excellent use of Markdown headers, bullet points, and formatting; highly        
                                   readable and well-organized.                                                    
 Tone & Style         5            Objective, professional, informative, and engaging.                             
 Safety & Ethics      5            Completely safe and benign technical educational content.                       
 Overall Average      5.0 / 5.0                                                                                    
                                                                                                                   

-------------------------------------------------------------------------------------------------------------------

🔴 Critical Issues                                                                                                 

None identified.                                                                                                   

🟡 Major Issues                                                                                                    

None identified.                                                                                                   

🟢 Minor Issues                                                                                                    

 • Issue: Specific index or survey names (e.g., TIOBE Index, Stack Overflow Developer Survey, GitHub Octoverse)    
   were omitted in favor of generic references ("search interest, open-source project activity, or developer survey
   responses").                                                                                                    
    • Recommendation: Briefly naming specific, well-known industry benchmarks (such as the Stack Overflow Developer
      Survey or TIOBE Index) would add explicit authority to the ranking baseline.                                 

-------------------------------------------------

Calling agent summarization_agent


Based on industry rankings, developer surveys (such as Stack Overflow and GitHub Octoverse), and search interest   
indices (like TIOBE), the top 3 programming languages overall are Python, JavaScript, and Java.                    

-------------------------------------------------------------------------------------------------------------------

1. Python 🐍                                                                                                       

 • Primary Use Cases: Artificial Intelligence & Machine Learning, Data Science & Analytics, Web Backend            
   Development, Automation & Scripting.                                                                            
 • Key Features:                                                                                                   
    • Simplicity and Readability: Python features an intuitive, highly readable syntax resembling plain English,   
      making it accessible for beginners while remaining powerful for experts.                                     
    • Rich Ecosystem: It boasts an extensive library ecosystem for AI and data processing (e.g., PyTorch,          
      TensorFlow, NumPy, Pandas) and robust web frameworks (Django, FastAPI).                                      
    • Interpreted & Dynamic: Python is dynamically typed and interpreted, enabling rapid prototyping and fast      
      iteration.                                                                                                   

-------------------------------------------------------------------------------------------------------------------

2. JavaScript 🟡                                                                                                   

 • Primary Use Cases: Frontend Web Development, Backend Web Services (Node.js), Mobile App Development (React      
   Native), Interactive Web Applications.                                                                          
 • Key Features:                                                                                                   
    • Language of the Web: JavaScript is natively supported by every modern web browser, making it essential for   
      building interactive user interfaces.                                                                        
    • Full-Stack Capability: With runtime environments like Node.js, developers can write both frontend and backend
      code using the same language.                                                                                
    • Event-Driven & Asynchronous: Built around an asynchronous, non-blocking event loop, JavaScript excels at     
      handling concurrent connections and real-time application events.                                            

-------------------------------------------------------------------------------------------------------------------

3. Java ☕                                                                                                         

 • Primary Use Cases: Enterprise Software Systems, Android App Development, Cloud-Native Microservices, Financial &
   High-Throughput Systems.                                                                                        
 • Key Features:                                                                                                   
    • "Write Once, Run Anywhere" (WORA): Java runs inside the Java Virtual Machine (JVM), allowing bytecode to     
      execute across any operating system without modification.                                                    
    • Strong Object-Oriented Structure: Java strictly enforces object-oriented programming principles, encouraging 
      maintainable architecture for large-scale enterprise projects.                                               
    • Performance & Stability: Featuring static compilation to bytecode, robust type checking, and mature automatic
      memory management (garbage collection), Java

Perform tests for searching.

In [15]:
print("================ Next golf events ===========================")
await run_prompt("Find the next 3 golf events to occur in the next month.")

================ Next golf events ===========================
Calling agent research_agent
Calling agent research_agent_tool
Calling agent research_agent


Here are the next 3 upcoming professional golf events:                                                             

1. Wyndham Championship (PGA Tour)                                                                                 

 • Dates: August 6–9                                                                                               
 • Location: Sedgefield Country Club — Greensboro, North Carolina                                                  
 • Overview: The final event of the PGA Tour regular season, where players compete for the final spots in the top  
   70 of the FedExCup standings to qualify for the playoffs.                                                       

-------------------------------------------------------------------------------------------------------------------

2. FedEx St. Jude Championship (PGA Tour)                                                                          

 • Dates: August 13–16                                                                                             
 • Location: TPC Southwind — Memphis, Tennessee                                                                    
 • Overview: The opening event of the three-leg PGA Tour FedExCup Playoffs, featuring the top 70 qualified players.

-------------------------------------------------------------------------------------------------------------------

3. The Standard Portland Classic (LPGA Tour)                                                                       

 • Dates: August 13–16                                                                                             
 • Location: Columbia Edgewater Country Club — Portland, Oregon                                                    
 • Overview: One of the longest-running non-major events on the LPGA Tour schedule, held concurrently alongside the
   DP World Tour's Danish Golf Championship (August 13–16 at Great Northern in Kerteminde, Denmark).

Calling agent critic_agent


🧾 Critique Report                                                                                                 

Agent Evaluated: Research Agent (research_agent)                                                                   
Original Request: Find the next 3 golf events to occur in the next month.                                          
Date of Review: May 19, 2024                                                                                       

-------------------------------------------------------------------------------------------------------------------

📊 Scores                                                                                                          

                                                                                                                   
 Dimension            Score (1–5)  Notes                                                                           
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
 Relevance            4/5          Directly addresses the request for upcoming professional golf events.           
 Accuracy             3/5          Dropped the year (2026) provided in the tool output, introducing temporal       
                                   ambiguity.                                                                      
 Reasoning & Logic    3/5          Combined two distinct tournaments into slot #3 to force a 3-item list format,   
                                   effectively listing 4 events.                                                   
 Completeness         4/5          Includes key details such as venue location, tour affiliation, dates, and       
                                   tournament significance.                                                        
 Structure & Clarity  4/5          Cleanly formatted with markdown headers, bolding, and bullet points.            
 Tone & Style         5/5          Professional, concise, and objective.                                           
 Safety & Ethics      5/5          Fully compliant with safety guidelines.                                         
 Overall Average      4.0                                                                                          
                                                                                                                   

-------------------------------------------------------------------------------------------------------------------

🔴 Critical Issues                                                                                                 

None identified.                                                                                                   

-------------------------------------------------------------------------------------------------------------------

🟡 Major Issues                                                                                                    

 • Issue: Arbitrary combining of two separate tournaments under entry #3.                                          
    • Why it matters: The user explicitly requested the next 3 golf events. By listing both The Standard Portland  
      Classic (LPGA) and the Danish Golf Championship (DP World Tour) under item #3, the response actually presents
      4 distinct events while mislabeling them as 3. This creates logical confusion.                               
    • Recommendation: Select exactly 3 distinct events sequentially (or pick 3 primary events across tours) and    
      mention concurrent tournaments in a brief secondary note or footnote rather than forcing two events into one 
      slot.                                                                                                        
 • Issue: Omission of the calendar year (2026) in the final output.                                                
    • Why it matters: The tool output specifically

Calling agent summarization_agent


Here are the next 3 upcoming professional golf events scheduled across the major tours:                            

-------------------------------------------------------------------------------------------------------------------

1. Wyndham Championship (PGA Tour)                                                                                 

 • Dates: August 6–9, 2026                                                                                         
 • Location: Sedgefield Country Club — Greensboro, North Carolina                                                  
 • Overview: The final event of the PGA Tour regular season, where players compete to secure a spot in the top 70  
   of the FedExCup standings and qualify for the playoffs.                                                         

-------------------------------------------------------------------------------------------------------------------

2. FedEx St. Jude Championship (PGA Tour)                                                                          

 • Dates: August 13–16, 2026                                                                                       
 • Location: TPC Southwind — Memphis, Tennessee                                                                    
 • Overview: The opening event of the three-leg PGA Tour FedExCup Playoffs, featuring the top 70 qualified players 
   from the regular season.                                                                                        

-------------------------------------------------------------------------------------------------------------------

3. The Standard Portland Classic (LPGA Tour)                                                                       

 • Dates: August 13–16, 2026                                                                                       
 • Location: Columbia Edgewater Country Club — Portland, Oregon                                                    
 • Overview: One of the longest-running non-major events on the LPGA Tour schedule.                                

-------------------------------------------------------------------------------------------------------------------


▌ Note on Concurrent Events: Taking place during the same weekend (August 13–16, 2026), the DP World Tour will   
▌ also host the Danish Golf Championship at Great Northern in Kerteminde, Denmark.                               